# Endüstriyel Uygulamalar / Industrial Applications

Bu notebook, bilgisayarlı görü modellerinin madencilik endüstrisindeki uygulamalarını gösterir:
1. **Defekt Tespiti**: Konveyor bantlarındaki kusurların tespiti
2. **Cevher Sınıflandırma**: Maden cevherlerinin sınıflandırılması

This notebook demonstrates computer vision applications in the mining industry:
1. **Defect Detection**: Detecting defects on conveyor belts
2. **Ore Classification**: Classifying mining ores

In [ ]:
# Gerekli kütüphaneleri içe aktar / Import required libraries
import os
import sys
from pathlib import Path

project_root = Path('../').resolve()
sys.path.insert(0, str(project_root))

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch

# Projedeki utility fonksiyonlarını import et
from services.utils import get_defect_colors, get_ore_class_colors, get_severity_level, calculate_anomaly_score

print("Kütüphaneler başarıyla içe aktarıldı!")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {DEVICE}")

## Türkçe Sınıf İsimleri / Turkish Class Names

In [ ]:
# Defekt sınıfları (Türkçe) / Defect classes (Turkish)
DEFECT_CLASSES = {
    0: "çatlak",      # Crack
    1: "çizik",       # Scratch
    2: "delik",       # Hole
    3: "leke",        # Stain
    4: "deformasyon"   # Deformation
}

# Cevher sınıfları (Türkçe) / Ore classes (Turkish)
ORE_CLASSES = {
    0: "manyetit",    # Magnetite
    1: "krom",        # Chromite
    2: "atık",        # Waste
    3: "düşük tenör"  # Low grade
}

# Renkleri al / Get colors
DEFECT_COLORS = get_defect_colors()
ORE_COLORS = get_ore_class_colors()

print("Defekt sınıfları:", list(DEFECT_CLASSES.values()))
print("Cevher sınıfları:", list(ORE_CLASSES.values()))

## Uygulama 1: Defekt Tespiti / Application 1: Defect Detection

Konveyor bantları üzerindeki madencilik ürünlerindeki defektleri tespit ediyoruz.
We detect defects on mining products on conveyor belts.

In [ ]:
# Test görüntüsü oluştur (defekt içeren) / Create test image with defects
def create_defect_test_image():
    """Defekt içeren test görüntüsü oluştur."""
    img = np.zeros((640, 640, 3), dtype=np.uint8)
    
    # Konveyor bant arka planı / Conveyor belt background
    img[:] = (70, 75, 80)
    # Bant çizgileri / Belt lines
    for i in range(0, 640, 40):
        cv2.line(img, (0, i), (640, i), (60, 65, 70), 2)
    
    # Cevher parçası 1 (normal) / Ore piece 1 (normal)
    cv2.circle(img, (150, 200), 60, (140, 100, 60), -1)
    cv2.circle(img, (150, 200), 60, (180, 140, 100), 3)
    
    # Cevher parçası 2 (çatlak var) / Ore piece 2 (with crack)
    cv2.circle(img, (350, 180), 70, (150, 110, 70), -1)
    cv2.circle(img, (350, 180), 70, (190, 150, 110), 3)
    # Çatlak / Crack
    cv2.line(img, (310, 140), (390, 220), (50, 50, 50), 3)
    cv2.line(img, (315, 145), (385, 215), (80, 80, 80), 1)
    
    # Cevher parçası 3 (delik var) / Ore piece 3 (with hole)
    cv2.circle(img, (520, 350), 55, (130, 90, 50), -1)
    cv2.circle(img, (520, 350), 55, (160, 120, 80), 3)
    # Delik / Hole
    cv2.circle(img, (530, 340), 15, (40, 40, 45), -1)
    
    # Cevher parçası 4 (çizik var) / Ore piece 4 (with scratch)
    cv2.circle(img, (180, 480), 50, (145, 105, 65), -1)
    cv2.circle(img, (180, 480), 50, (175, 135, 95), 3)
    # Çizik / Scratch
    cv2.line(img, (140, 460), (220, 500), (60, 60, 60), 2)
    
    # Atık / Waste
    cv2.rectangle(img, (400, 450), (550, 580), (100, 100, 100), -1)
    cv2.rectangle(img, (400, 450), (550, 580), (120, 120, 120), 3)
    
    return img

defect_image = create_defect_test_image()
defect_image_pil = Image.fromarray(defect_image)

plt.figure(figsize=(12, 8))
plt.imshow(defect_image)
plt.title("Defekt Tespiti Test Görüntüsü / Defect Detection Test Image")
plt.axis('off')
plt.show()

In [ ]:
# YOLO modeli yükle / Load YOLO model
from ultralytics import YOLO

print("YOLO modeli yükleniyor (defekt tespiti için)...")
# Gerçek uygulamada: yolo11s-seg.pt kullanın
# Burada test için yolo11n.pt kullanıyoruz
defect_model = YOLO("yolo11n.pt")
defect_model.to(DEVICE)
print("Model yüklendi!")

In [ ]:
# Defekt tespiti simülasyonu / Simulate defect detection
# Gerçek model olmadan simüle edilmiş sonuçlar
def simulate_defect_detection(image):
    """
    Defekt tespiti simülasyonu.
    Gerçek uygulamada YOLO model kullanılır.
    """
    detections = []
    
    # Simüle edilmiş tespitler / Simulated detections
    # (class_id, confidence, bbox)
    detections.append({
        "label": "manyetit",
        "confidence": 0.92,
        "bbox": [90, 140, 210, 260]
    })
    
    detections.append({
        "label": "çatlak",
        "confidence": 0.85,
        "bbox": [280, 110, 420, 250]
    })
    
    detections.append({
        "label": "delik",
        "confidence": 0.88,
        "bbox": [465, 295, 585, 405]
    })
    
    detections.append({
        "label": "çizik",
        "confidence": 0.78,
        "bbox": [130, 430, 230, 530]
    })
    
    detections.append({
        "label": "atık",
        "confidence": 0.95,
        "bbox": [400, 450, 550, 580]
    })
    
    return detections

# Tespit yap / Run detection
defect_detections = simulate_defect_detection(defect_image)
print(f"Tespit edilen nesne sayısı: {len(defect_detections)}")

for det in defect_detections:
    print(f"  - {det['label']:15s} | Güven: {det['confidence']:.2f}")

In [ ]:
# Sonuçları görselleştir / Visualize results
def visualize_defect_results(image, detections):
    """Defekt tespit sonuçlarını görselleştir."""
    img = image.copy()
    
    # Renk eşleştirmesi / Color mapping
    color_map = {
        "manyetit": (180, 80, 50),
        "krom": (50, 180, 50),
        "atık": (128, 128, 128),
        "düşük tenör": (50, 100, 200),
        "çatlak": (0, 0, 255),
        "çizik": (255, 165, 0),
        "delik": (128, 0, 128),
        "leke": (255, 255, 0),
        "deformasyon": (255, 0, 128)
    }
    
    for det in detections:
        x1, y1, x2, y2 = det["bbox"]
        color = color_map.get(det["label"], (255, 255, 255))
        
        # Bounding box çiz / Draw bounding box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        
        # Etiket yaz / Write label
        label = f"{det['label']}: {det['confidence']:.2f}"
        cv2.rectangle(img, (x1, y1-25), (x1+len(label)*10, y1), color, -1)
        cv2.putText(img, label, (x1+5, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    return img

result_img = visualize_defect_results(defect_image, defect_detections)

plt.figure(figsize=(14, 10))
plt.imshow(result_img)
plt.title("Defekt Tespiti Sonuçları / Defect Detection Results")
plt.axis('off')
plt.show()

In [ ]:
# Anomali skoru hesapla / Calculate anomaly score
defect_confidences = [d["confidence"] for d in defect_detections if d["label"] in ["çatlak", "çizik", "delik", "leke", "deformasyon"]]
anomaly_score = calculate_anomaly_score(defect_confidences) if defect_confidences else 0
severity = get_severity_level(anomaly_score)

print("=== Defekt Analiz Sonuçları / Defect Analysis Results ===\n")
print(f"Tespit edilen toplam nesne: {len(defect_detections)}")
print(f"Defekt sayısı: {len(defect_confidences)}")
print(f"Anomali skoru: {anomaly_score:.1f}/100")
print(f"Şiddet seviyesi: {severity}")

# Defekt özeti / Defect summary
print("\nDefekt dağılımı / Defect distribution:")
for label in ["çatlak", "çizik", "delik", "leke", "deformasyon"]:
    count = sum(1 for d in defect_detections if d["label"] == label)
    if count > 0:
        print(f"  - {label}: {count}")

## Uygulama 2: Cevher Sınıflandırma / Application 2: Ore Classification

Maden cevherlerini sınıflandırıyoruz: Manyetit, Krom, Atık, Düşük Tenör.
We classify mining ores: Magnetite, Chromite, Waste, Low Grade.

In [ ]:
# Test görüntüsü oluştur (cevher sınıflandırma) / Create test image for ore classification
def create_ore_test_image():
    """Cevher sınıflandırma test görüntüsü oluştur."""
    img = np.zeros((640, 640, 3), dtype=np.uint8)
    
    # Arka plan / Background
    img[:] = (60, 65, 70)
    
    # Manyetit (koyu kırmızımsı kahverengi) / Magnetite (dark reddish brown)
    cv2.circle(img, (120, 120), 55, (100, 45, 35), -1)
    cv2.circle(img, (120, 120), 55, (140, 85, 75), 3)
    cv2.putText(img, "M", (105, 125), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    # Krom (yeşilimsi) / Chromite (greenish)
    cv2.circle(img, (300, 100), 50, (60, 120, 50), -1)
    cv2.circle(img, (300, 100), 50, (100, 160, 90), 3)
    cv2.putText(img, "K", (285, 105), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    # Atık (gri) / Waste (gray)
    cv2.circle(img, (480, 150), 45, (130, 130, 130), -1)
    cv2.circle(img, (480, 150), 45, (160, 160, 160), 3)
    cv2.putText(img, "A", (465, 155), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    # Düşük tenör (soluk renk) / Low grade (pale color)
    cv2.circle(img, (180, 320), 48, (150, 140, 120), -1)
    cv2.circle(img, (180, 320), 48, (180, 170, 150), 3)
    cv2.putText(img, "D", (165, 325), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    # Manyetit 2
    cv2.circle(img, (420, 300), 52, (95, 40, 30), -1)
    cv2.circle(img, (420, 300), 52, (135, 80, 70), 3)
    cv2.putText(img, "M", (405, 305), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    # Krom 2
    cv2.circle(img, (100, 500), 47, (55, 115, 45), -1)
    cv2.circle(img, (100, 500), 47, (95, 155, 85), 3)
    cv2.putText(img, "K", (85, 505), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    # Atık 2
    cv2.circle(img, (320, 480), 50, (125, 125, 125), -1)
    cv2.circle(img, (320, 480), 50, (155, 155, 155), 3)
    cv2.putText(img, "A", (305, 485), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    # Düşük tenör 2
    cv2.circle(img, (520, 480), 45, (145, 135, 115), -1)
    cv2.circle(img, (520, 480), 45, (175, 165, 145), 3)
    cv2.putText(img, "D", (505, 485), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    return img

ore_image = create_ore_test_image()
ore_image_pil = Image.fromarray(ore_image)

plt.figure(figsize=(12, 8))
plt.imshow(ore_image)
plt.title("Cevher Sınıflandırma Test Görüntüsü / Ore Classification Test Image")
plt.axis('off')
plt.show()

print("M = Manyetit, K = Krom, A = Atık, D = Düşük Tenör")

In [ ]:
# Cevher sınıflandırma simülasyonu / Simulate ore classification
def simulate_ore_classification(image):
    """
    Cevher sınıflandırma simülasyonu.
    Gerçek uygulamada YOLO segmentasyon modeli kullanılır.
    """
    classifications = []
    
    # Simüle edilmiş sınıflandırmalar / Simulated classifications
    classifications.append({
        "label": "manyetit",
        "confidence": 0.94,
        "bbox": [65, 65, 175, 175],
        "area_percent": 12.5
    })
    
    classifications.append({
        "label": "krom",
        "confidence": 0.91,
        "bbox": [250, 50, 350, 150],
        "area_percent": 10.0
    })
    
    classifications.append({
        "label": "atık",
        "confidence": 0.96,
        "bbox": [435, 105, 525, 195],
        "area_percent": 8.0
    })
    
    classifications.append({
        "label": "düşük tenör",
        "confidence": 0.88,
        "bbox": [132, 272, 228, 368],
        "area_percent": 9.0
    })
    
    classifications.append({
        "label": "manyetit",
        "confidence": 0.92,
        "bbox": [368, 248, 472, 352],
        "area_percent": 10.5
    })
    
    classifications.append({
        "label": "krom",
        "confidence": 0.89,
        "bbox": [53, 453, 147, 547],
        "area_percent": 8.5
    })
    
    classifications.append({
        "label": "atık",
        "confidence": 0.94,
        "bbox": [270, 430, 370, 530],
        "area_percent": 10.0
    })
    
    classifications.append({
        "label": "düşük tenör",
        "confidence": 0.85,
        "bbox": [475, 435, 565, 525],
        "area_percent": 7.8
    })
    
    return classifications

# Sınıflandırma yap / Run classification
ore_classifications = simulate_ore_classification(ore_image)
print(f"Sınıflandırılan nesne sayısı: {len(ore_classifications)}")

In [ ]:
# Cevher sonuçlarını görselleştir / Visualize ore results
def visualize_ore_results(image, classifications):
    """Cevher sınıflandırma sonuçlarını görselleştir."""
    img = image.copy()
    
    # Cevher renkleri / Ore colors
    ore_colors = {
        "manyetit": (50, 50, 200),      # Kırmızımsı / Reddish
        "krom": (50, 200, 50),          # Yeşil / Green
        "atık": (150, 150, 150),        # Gri / Gray
        "düşük tenör": (200, 150, 50)   # Turuncu / Orange
    }
    
    for cls in classifications:
        x1, y1, x2, y2 = cls["bbox"]
        color = ore_colors.get(cls["label"], (255, 255, 255))
        
        # Bounding box çiz / Draw bounding box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
        
        # Etiket yaz / Write label
        label = f"{cls['label']}: {cls['confidence']:.2f}"
        cv2.rectangle(img, (x1, y1-30), (x1+len(label)*9, y1), color, -1)
        cv2.putText(img, label, (x1+3, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 2)
    
    return img

ore_result_img = visualize_ore_results(ore_image, ore_classifications)

plt.figure(figsize=(14, 10))
plt.imshow(ore_result_img)
plt.title("Cevher Sınıflandırma Sonuçları / Ore Classification Results")
plt.axis('off')
plt.show()

In [ ]:
# Cevher metriklerini hesapla / Calculate ore metrics
def calculate_ore_metrics(classifications):
    """Cevher metriklerini hesapla."""
    # Sınıf bazında alan yüzdeleri / Area percentages per class
    class_areas = {}
    for cls in classifications:
        label = cls["label"]
        if label not in class_areas:
            class_areas[label] = 0
        class_areas[label] += cls["area_percent"]
    
    total_area = sum(class_areas.values())
    
    # Yüzdeleri hesapla / Calculate percentages
    percentages = {k: (v/total_area)*100 for k, v in class_areas.items()}
    
    return percentages

ore_percentages = calculate_ore_metrics(ore_classifications)

print("=== Cevher Analiz Sonuçları / Ore Analysis Results ===\n")
print("Sınıflandırma dağılımı / Classification distribution:")
for label, pct in sorted(ore_percentages.items()):
    print(f"  - {label:15s}: %{pct:.1f}")

# Metal oranı hesapla / Calculate metal ratio
valuable = ore_percentages.get("manyetit", 0) + ore_percentages.get("krom", 0)
waste = ore_percentages.get("atık", 0)
low_grade = ore_percentages.get("düşük tenör", 0)

if waste + low_grade > 0:
    metal_ratio = valuable / (waste + low_grade)
else:
    metal_ratio = float('inf')

print(f"\nDeğerli cevher oranı: %{valuable:.1f}")
print(f"Atık oranı: %{waste:.1f}")
print(f"Düşük tenör oranı: %{low_grade:.1f}")
print(f"Metal/Atık oranı: {metal_ratio:.2f}")

In [ ]:
# Pasta grafiği ile göster / Show with pie chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pasta grafiği / Pie chart
labels = list(ore_percentages.keys())
values = list(ore_percentages.values())
colors_pie = ['#3232FF', '#32FF32', '#969696', '#C89600']

axes[0].pie(values, labels=labels, autopct='%1.1f%%', colors=colors_pie, startangle=90)
axes[0].set_title("Cevher Dağılımı / Ore Distribution")

# Bar grafiği / Bar chart
x = np.arange(len(labels))
bars = axes[1].bar(x, values, color=colors_pie)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=15)
axes[1].set_ylabel('Alan Yüzdesi / Area Percentage (%)')
axes[1].set_title("Cevher Yüzdeleri / Ore Percentages")
axes[1].grid(axis='y', alpha=0.3)

# Değerleri üzerine yaz / Write values on bars
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                f'%{val:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Özet / Summary

Bu notebook'ta iki temel endüstriyel uygulamayı gördük:

### 1. Defekt Tespiti:
- Konveyor bantlarındaki kusurları tespit eder
- Çatlak, çizik, delik, leke, deformasyon gibi defektleri tanır
- Anomali skoru ve şiddet seviyesi hesaplar

### 2. Cevher Sınıflandırma:
- Manyetit, krom, atık ve düşük tenör cevherlerini sınıflandırır
- Her cevher türünün alan yüzdesini hesaplar
- Metal/atık oranını hesaplayarak kalite kontrolü sağlar

### Gerçek Uygulama:
Gerçek uygulamada bu sistemler YOLO11 veya özel eğitilmiş modeller kullanılarak entegre edilir.
Gerçek zamanlı tespit için GPU kullanılması önerilir.

In [ ]:
# Sistem özeti / System summary
print("=" * 60)
print("ENDÜSTRİYEL GÖRÜNTÜ İŞLEME SİSTEMİ ÖZETİ")
print("INDUSTRIAL IMAGE PROCESSING SYSTEM SUMMARY")
print("=" * 60)

print("\n--- Defekt Tespiti / Defect Detection ---")
print(f"Tespit edilen nesne: {len(defect_detections)}")
print(f"Defekt sayısı: {len(defect_confidences)}")
print(f"Anomali skoru: {anomaly_score:.1f}/100")
print(f"Şiddet: {severity}")

print("\n--- Cevher Sınıflandırma / Ore Classification ---")
for label, pct in sorted(ore_percentages.items()):
    print(f"  {label:15s}: %{pct:.1f}")
print(f"\nMetal/Atık oranı: {metal_ratio:.2f}")

print("\n" + "=" * 60)
print("Sistem hazır! / System ready!")
print("=" * 60)